# Katube Colab - Download de Audio do YouTube

Sistema modular para download de audio com persistencia no Google Drive e exportacao de metadados em CSV

In [ ]:
# Clone do repositorio
!git clone https://github.com/seu-usuario/katube-colab.git
%cd katube-colab

In [ ]:
# Instala dependencias
!pip install -q -r requirements.txt

## Monta Google Drive

In [ ]:
from downloaders import DriveManager
from config import Config

# Monta Drive
drive_manager = DriveManager()
drive_manager.mount_drive()

# Cria estrutura de pastas
drive_manager.setup_folder_structure()

# Verifica espaco
space_info = drive_manager.check_drive_space()
if space_info['available']:
    print(f"Espaco livre no Drive: {space_info['free_formatted']}")

## Configuracao (Opcional)

Personalize as configuracoes se necessario

In [ ]:
# Exibe configuracao atual
Config.print_config()

# Para alterar configuracoes:
# Config.AUDIO_FORMAT = 'mp3'  # Opcoes: 'mp3', 'flac', 'wav', 'm4a', 'opus'
# Config.AUDIO_QUALITY = 320
# Config.DELAY_MIN = 15
# Config.DELAY_MAX = 25
# Config.CSV_ENABLED = True  # Ativar/desativar geracao de CSV
# Config.CHECKPOINT_ENABLED = True  # Ativar/desativar checkpoint

## Download com CSV - Video Individual

Baixa o audio e gera CSV com todos os metadados

In [ ]:
from downloaders import YouTubeDownloader

downloader = YouTubeDownloader()

# Substitua pela URL do video
url = "https://www.youtube.com/watch?v=VIDEO_ID"

result = downloader.download_from_url(url)

if result['success']:
    print("Download concluido!")
    print(f"Arquivo: {result.get('file', 'N/A')}")
    if result.get('csv_path'):
        print(f"CSV gerado: {result['csv_path']}")
else:
    print(f"Erro: {result.get('error', 'Desconhecido')}")

## Download com CSV - Playlist

Baixa todos os audios da playlist e gera CSV unificado com metadados

In [ ]:
# Substitua pela URL da playlist
playlist_url = "https://www.youtube.com/playlist?list=PLAYLIST_ID"

result = downloader.download_from_url(playlist_url)

if result['success']:
    stats = result['stats']
    print(f"Playlist concluida!")
    print(f"Total: {stats['total_attempted']}")
    print(f"Sucesso: {stats['successful']}")
    print(f"Falhas: {stats['failed']}")
    print(f"Pulados: {stats['skipped']}")
    
    if result.get('csv_path'):
        print(f"\nCSV gerado: {result['csv_path']}")
        print("\nPara baixar o CSV, use:")
        print(f"from google.colab import files")
        print(f"files.download('{result['csv_path']}')")

## Download com CSV - Canal

Baixa todos os audios do canal e gera CSV unificado

In [ ]:
# Substitua pela URL do canal
channel_url = "https://www.youtube.com/@username"

result = downloader.download_from_url(channel_url)

if result['success']:
    stats = result['stats']
    print(f"Canal concluido!")
    print(f"Sucesso: {stats['successful']}")
    
    if result.get('csv_path'):
        print(f"\nCSV gerado: {result['csv_path']}")

## Download com CSV - Arquivo TXT

Processa multiplas URLs de um arquivo txt e gera CSV unificado

In [ ]:
# Cria arquivo txt de exemplo
with open('urls.txt', 'w') as f:
    f.write("https://www.youtube.com/watch?v=VIDEO_ID_1\n")
    f.write("https://www.youtube.com/watch?v=VIDEO_ID_2\n")
    f.write("https://www.youtube.com/watch?v=VIDEO_ID_3\n")

# Download
result = downloader.download_from_txt('urls.txt')

if result['success']:
    print(f"TXT concluido! ID: txt_{result['txt_id']}")
    stats = result['stats']
    print(f"Sucesso: {stats['successful']}/{result['total_urls']}")
    
    if result.get('csv_path'):
        print(f"\nCSV gerado: {result['csv_path']}")

## Visualizar CSV de Metadados

Preview dos metadados extraidos

In [ ]:
import pandas as pd
from pathlib import Path

# Substitua pelo caminho do CSV gerado
csv_path = "/content/drive/MyDrive/Katube_Download/Playlist_PLxxx/metadados.csv"

if Path(csv_path).exists():
    df = pd.read_csv(csv_path, sep='|')
    
    print(f"Total de videos: {len(df)}")
    print(f"\nPrimeiras linhas:")
    display(df.head())
    
    print(f"\nColunas disponiveis:")
    print(df.columns.tolist())
    
    print(f"\nEstatisticas de download:")
    print(df['download_status'].value_counts())
else:
    print("Arquivo CSV nao encontrado")

## Baixar CSV para seu Computador

Faz download do arquivo CSV gerado

In [ ]:
from google.colab import files

# Substitua pelo caminho do CSV gerado
csv_path = "/content/drive/MyDrive/Katube_Download/Playlist_PLxxx/metadados.csv"

if Path(csv_path).exists():
    files.download(csv_path)
    print("Download iniciado!")
else:
    print("Arquivo CSV nao encontrado")

## Estatisticas e Resumo

In [ ]:
# Resumo dos downloads
summary = drive_manager.get_download_summary()

if summary['available']:
    print("RESUMO GERAL")
    print("="*50)
    print(f"Total de pastas: {summary['total_folders']}")
    print(f"Total de arquivos: {summary['total_files']}")
    print(f"Tamanho total: {summary['total_size_formatted']}")
    print("\nPOR TIPO:")
    print(f"  Videos: {summary['by_type']['video']}")
    print(f"  Playlists: {summary['by_type']['playlist']}")
    print(f"  Canais: {summary['by_type']['channel']}")
    print(f"  TXT: {summary['by_type']['txt']}")
else:
    print(f"Erro: {summary.get('error', 'Desconhecido')}")

## Verificar Progresso de Download (Checkpoint)

Visualiza o progresso de downloads em andamento ou interrompidos

In [ ]:
import json
from pathlib import Path

# Substitua pelo caminho da pasta de download
download_path = Path("/content/drive/MyDrive/Katube_Download/Playlist_PLxxx")
checkpoint_path = download_path / "checkpoint.json"

if checkpoint_path.exists():
    with open(checkpoint_path, 'r') as f:
        checkpoint = json.load(f)
    
    print("INFORMACOES DO CHECKPOINT")
    print("="*50)
    print(f"URL: {checkpoint.get('url', 'N/A')}")
    print(f"Total de videos: {checkpoint.get('total_videos', 0)}")
    print(f"Processados: {len(checkpoint.get('processed', []))}")
    print(f"Falhados: {len(checkpoint.get('failed', []))}")
    print(f"Ultima atualizacao: {checkpoint.get('last_update', 'N/A')}")
    
    pendentes = checkpoint.get('total_videos', 0) - len(checkpoint.get('processed', []))
    print(f"\nPendentes: {pendentes}")
    
    if pendentes > 0:
        print("\nVoce pode retomar o download executando a celula de download novamente!")
else:
    print("Nenhum checkpoint encontrado")

## Limpeza (Opcional)

In [ ]:
# Remove pastas vazias
removed = drive_manager.cleanup_empty_folders()
print(f"Pastas vazias removidas: {removed}")

## Verificar Logs

In [ ]:
# Lista arquivos de log
log_path = Config.get_log_path()

if log_path.exists():
    print("LOGS DISPONIVEIS:")
    for log_file in log_path.glob('*.log'):
        size = log_file.stat().st_size / 1024
        print(f"  {log_file.name} ({size:.2f} KB)")
else:
    print("Nenhum log encontrado")